# Test

In [ ]:
def amongus():
    print("AMONGUS")

In [ ]:
import uproot
import awkward as ak
import numpy as np
import scipy
import scipy.stats as sp
import matplotlib
import matplotlib.pyplot as plt
import math as math
import hist
import vector
import mplhep
print("uproot version",uproot.__version__)
print("awkward version",ak.__version__)
print("numpy version",np.__version__)
print("scipy version",scipy.__version__)
print("matplotlib version",matplotlib.__version__)
print("hist version",hist.__version__)
print("vector version",vector.__version__)
print("mplhep version",mplhep.__version__)

Ouverture des fichiers

In [ ]:
pathOnline = "../data/counters.online.csv"
pathOffline = "../data/counters.offline.csv"
fileOnline =  open(pathOnline, 'r')
fileOffline = open(pathOffline, 'r')

In [ ]:
def read_file(file):
    t = file.readlines()
    r = {}
    k = []
    for i, lt in enumerate(t):
        lt = lt.rstrip('\n')       # get rid off linebreaks
        lt = lt.split(',')         # separate values
        for j in range(len(lt)):
            if i == 0:
                k = lt
                r[lt[j]] = []
            else:
                r[k[j]].append(int(lt[j]))
    return ak.Array(r)

In [ ]:
dOn = read_file(fileOnline)    #data Online
dOff = read_file(fileOffline)  #data Offline

On ne considère ici que certains runs

In [ ]:
runs = [290293,290401, 291263]
allRuns = dOff.run # tous les runs

fonctions pour le calcul de N_MB

In [ ]:
def F_online(pRuns):
    out = {"Fi": [], "erri": [], "duri": []}
    for r in pRuns:
        cd = dOn.run == r
        erri = np.sqrt( 1/dOn.cint7l0b[cd] + 1/dOn.cmul7l0b[cd] )[0] #\sqrt{(\sqrt cint7 / cint7)^2 + (\sqrt cmul7 / cmul7)^2} pour une loi de poisson
        Fi = (dOn.cint7l0b[cd] / dOn.cmul7l0b[cd])[0]
        
        out["Fi"].append(Fi)
        out["erri"].append(erri*Fi)
        out["duri"].append(dOn["duration(s)"][cd][0])
    return out

def avgdF_on(pFon):
    res = 0
    err = 0
    for i in range(len(pFon["Fi"])):
        res += pFon["Fi"][i]*pFon["duri"][i]/sum(pFon["duri"])
        err += pow(pFon["erri"][i]/pFon["Fi"][i],2) * pow(pFon["duri"][i],2) / pow(sum(pFon["duri"]),2)
    
    return [res, np.sqrt(err)*res]

rFon = F_online(allRuns)
print(avgdF_on(rFon))

In [ ]:
def F_offline(pRuns):
    fs = {}
    valMu = {}
    for r in pRuns:
        cd = dOff.run == r
        
        fLHC = 11.245e3 # Hz
        L0b_rate = dOn.cint7l0b[cd] / dOn["duration(s)"][cd] # 1/s
        R_PS = dOff.cint7ps[cd] / dOff.cint7all[cd] # taux d'évenements passant la physics selection
        mu = -np.log(1-( R_PS * L0b_rate / (dOn.interacting_bunches[cd] * fLHC))[0])
        valMu[r] = mu
        F_PU = mu/(1-np.exp(-mu))    # correction de pileup
        
        fs[r] = (dOff.cmsl7all[cd] / dOff["cmsl7all&0mul"][cd] \
                *dOff.cint7all[cd] / dOff["cint7all&0msl"][cd] \
                *F_PU )[0]
    return fs

In [ ]:
lFoff = F_offline(allRuns)

plt.plot(lFoff.values())

## Erreur statistique

Fit statistique des runs - Gaussien ne semble pas correct pour le facteur Online

In [ ]:
def FitRuns(factors: dict, plot: bool=False, doprint: bool=False):
    vals = list(factors.values())
    res = sp.fit(sp.norm, vals, [[min(vals), max(vals)],[0,max(vals)-min(vals)]])
    if doprint: print(res.params); print("success: ", res.success)
    if plot:
        res.plot()
        plt.show()
    return res.params

#FitRuns(F_online(allRuns), True, True)
FitRuns(F_offline(allRuns), True, True)